# 04 — Controlled SQL Analytics

This notebook is intentionally small. Focus on one concept before moving to the next.

In [1]:
from pathlib import Path
import sys
import json

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
print('Project root:', ROOT)

Project root: /mnt/data/raa_latest_work/reservation-analytics-ai-agent


## Goal

Understand why the application resolves Campaign + Product + Country before executing controlled analytics SQL.

In [2]:
from app.settings import load_settings
from app.data.backend import create_backend
from app.analytics.resolver import CampaignResolver
from app.analytics.service import AnalyticsService
from app.core.models import ReservationQuery

settings = load_settings('config/local.env')
backend = create_backend(settings)
resolver = CampaignResolver(backend)
analytics = AnalyticsService(backend)

In [3]:
query = ReservationQuery(
    country='Germany',
    product='Phone Mi 17 Pro',
    campaign_id='CMP001',
)
campaigns = resolver.resolve(query)
print(json.dumps([item.model_dump() for item in campaigns], indent=2, ensure_ascii=False))


[
  {
    "campaign_id": "CMP001",
    "campaign_name": "Phone Mi 17 Pro Launch",
    "product_id": "P001",
    "product_name": "Phone Mi 17 Pro",
    "country_code": "DE",
    "country_name": "Germany",
    "start_time": "2026-08-01 00:00:00",
    "end_time": "2026-08-15 23:59:59"
  }
]


In [4]:
campaign = campaigns[0]
print(analytics.run('reserved_users', campaign))
print(analytics.run('conversion_rate', campaign))

CMP001 — Phone Mi 17 Pro Launch (Germany): 8 reserved users.
CMP001 — Phone Mi 17 Pro Launch (Germany): reservation-to-order conversion rate was 62.50%.


Key design rule:

- LLM: understand user intent and context.
- Application: select controlled SQL.
- Data Mart: provide trusted metrics or detail records.